In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))  # Add parent directory to path

import numpy as np
from diffusion_geometry.visualisation import *
from plotly.subplots import make_subplots

from generate_data import gen_2d_data, load_image_point_cloud
from diffusion_geometry import DiffusionGeometry

from methods.pde import *

Load tissue slice image and convert to point cloud

In [ ]:
data1 = load_image_point_cloud("../data/tissue_slice.jpg", n=5000, threshold=0.7, intensity_weighted=False)

plot_scatter_2d(data1, color = 'black').show()
dg1 = DiffusionGeometry.from_point_cloud(data1, n_function_basis = 50)

Set up diffusion geometry: construct the Laplacian.

In [ ]:
from diffusion_geometry import DiffusionGeometry

f = np.linalg.norm(data1 - [-0.6,-0.2], axis=1) < 0.4
f = dg1.function(f)
f.coeffs[20:] = 0

plot_scatter_2d(data1, color = f.to_ambient()).show()

Solve the heat equation

In [ ]:
def heat_figs(dg, f, t_end, num_frames, scatter_size = 3):
    t_values = np.linspace(0, t_end, num_frames)
    operator = - dg.laplacian(0)
    ft = solve_differential_operator(operator, f, t_values)
    amax = np.absolute(f.to_ambient()).max()
    figs = []
    for i in range(len(t_values)):
        fig = plot_scatter_2d(dg.cache.data_matrix, color = ft[i].to_ambient(), size = scatter_size, amax=amax)
        figs.append(fig)
    return figs

Solve the wave equation

In [ ]:
from diffusion_geometry.operators import block, zero, identity

def damped_wave_figs(dg, f, t_end=2, num_frames=5, damping=0, scatter_size = 3):

    # Define the space of pairs of functions as a direct sum
    function_pairs_space = dg1.function_space + dg1.function_space

    # Define the initial value and speed
    initial_value = f
    initial_speed = dg1.function_space.zeros()
    initial_pair = function_pairs_space.pack(initial_value, initial_speed)

    # Define the damped wave operator
    operator = block([[zero(dg1.function_space), identity(dg1.function_space)],
                    [-dg1.laplacian(0), -damping*identity(dg1.function_space)]])

    # Solve the differential operator
    t_values = np.linspace(0, t_end, num_frames)
    ft = solve_differential_operator(operator, initial_pair, t_values)
    values, speeds = ft.unpack()
    values = values.real

    # Create plots at different time frames
    amax = 1 * np.absolute(values[0].to_ambient()).max()
    plots = []
    for function in values:
        plots.append(plot_scatter_2d(dg.cache.data_matrix, color=function.to_ambient(), amax=amax, size = scatter_size))
    return plots

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

num_frames = 6

fig = make_subplots(
    rows=num_frames,
    cols=3,
    # specs=specs,
    horizontal_spacing=0.02,
    vertical_spacing=0.02,
)

scatter_size = 3
end_time = 5
# --- Add all traces ---
heat = heat_figs(dg1, f, t_end=0.3, num_frames=num_frames, scatter_size=scatter_size)
for i, subfig in enumerate(heat):
    fig.add_traces(list(subfig.data), rows=[i+1]*len(subfig.data), cols=[1]*len(subfig.data))

wave = damped_wave_figs(dg1, f, t_end=5, num_frames=num_frames, damping=0, scatter_size=scatter_size)
for i, subfig in enumerate(wave):
    fig.add_traces(list(subfig.data), rows=[i+1]*len(subfig.data), cols=[2]*len(subfig.data))

damped = damped_wave_figs(dg1, f, t_end=4, num_frames=num_frames, damping=0.5, scatter_size=scatter_size)
for i, subfig in enumerate(damped):
    fig.add_traces(list(subfig.data), rows=[i+1]*len(subfig.data), cols=[3]*len(subfig.data))

clean_fig(fig)

# --- Synchronize 2D axis ranges (rows 1–2) ---
all_figs_2d = heat + wave + damped
x_vals, y_vals = [], []
for subfig in all_figs_2d:
    for t in subfig.data:
        if hasattr(t, "x") and t.x is not None:
            x_vals.extend([v for v in t.x if v is not None])
        if hasattr(t, "y") and t.y is not None:
            y_vals.extend([v for v in t.y if v is not None])
xrange = [min(x_vals), max(x_vals)]
yrange = [min(y_vals), max(y_vals)]
fig.update_xaxes(range=xrange)
fig.update_yaxes(range=yrange)


# --- Layout cleanup ---
fig.update_layout(width=1000, height=1300)
fig.update_layout(margin=dict(l=0, r=0, t=0, b=0))

fig.show()
fig.write_image("figs/new_output.pdf", scale=1)


In [ ]:
def labels(row, col):
    if row == 1:
        if col == 1:
            return r"$\frac{\partial u_t}{\partial t} = - \Delta u_t$"
        elif col == 2:
            return r"$\frac{\partial^2 u_t}{\partial t^2} = - \Delta u_t$"
        elif col == 3:
            return r"$\frac{\partial^2 u_t}{\partial t^2} = - \Delta u_t - 0.1 \frac{\partial u_t}{\partial t}$"
    else:
        return

overpic_labels(fig, labels, 
               stretch_x=1, 
               stretch_y=1,
               offset_y=-10,
            #    offset_x=5,
               )


In [ ]:
# Generate grid of point in 2d:
grid_x, grid_y = np.meshgrid(np.linspace(-1, 1, 50), np.linspace(-1, 1, 50))
data = np.vstack([grid_x.ravel(), grid_y.ravel()]).T
data = np.random.uniform(-1, 1, size=(2500, 2))
dg2 = DiffusionGeometry.from_point_cloud(data, n_function_basis = 50)
plot_scatter_2d(data, color = 'black').show()

In [ ]:
f = np.exp(-5 * ((data[:,0] + 0.8)**2 + (data[:,1] - 0.3)**2)) - 0.5*np.exp(-5 * ((data[:,0] - 0.8)**2 + (data[:,1] + 0.3)**2))
f = dg2.function(f)
plot_scatter_2d(data, color = f.to_ambient()).show()

In [ ]:
import plotly.graph_objects as go
import visualisation

def plot_3d_mesh(data, f, camera=None, amax=None):
    fig = go.Figure(data=[go.Mesh3d(x=data[:,0],
                                    y=data[:,1],
                                    z=f.to_ambient(),
                                    intensity=f.to_ambient(),
                                    colorscale=visualisation.DEFAULT_COLORSCALE,
                                    cmid=0,
                                    cmin=-amax if amax is not None else None,
                                    cmax=amax if amax is not None else None,
                                    opacity=1,
                                    lighting=dict(
    ambient=1,    # raise base brightness (0.6–0.8)
    diffuse=1,    # keep some directional contrast
    specular=0.3,   # low, to avoid grey highlights
    roughness=0.8,  # smoother highlights (reduce dull reflection)
    fresnel=1.    # minimal edge brightening
)

                                    )])
    visualisation.clean_fig(fig)
    if camera is not None:
        fig.update_layout(scene_camera=camera)
    return fig

camera = dict(
    eye=dict(x=0.9, y=-1.7, z=0.9),
    up=dict(x=0., y=0., z=1.),
    center=dict(x=-0.05, y=0, z=-0.1),
)

fig = plot_3d_mesh(data, f, camera=camera)
fig.show()

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

num_frames = 5
t_end = 0.2

specs = [
    [{"type": "xy"}]*num_frames,
    [{"type": "scene"}]*num_frames,
]

fig = make_subplots(
    rows=2,
    cols=num_frames,
    specs=specs,
    horizontal_spacing=0.02,
    vertical_spacing=0.02,
)

scatter_size = 3
fmin = f.to_ambient().min()
fmax = f.to_ambient().max()
amax=max(abs(fmin), abs(fmax))

# --- Add all traces ---
t_values = np.linspace(0, t_end, num_frames)
operator = - dg2.laplacian(0)
ft = solve_differential_operator(operator, f, t_values)
for i in range(len(t_values)):
    scatter_fig = plot_scatter_2d(dg2.cache.data_matrix, color = ft[i].to_ambient(), size = scatter_size, amax=amax)
    fig.add_traces(list(scatter_fig.data), rows=[1]*len(scatter_fig.data), cols=[i+1]*len(scatter_fig.data))
    mesh_fig = plot_3d_mesh(data, ft[i], camera=camera, amax=amax)
    fig.add_traces(list(mesh_fig.data), rows=[2]*len(mesh_fig.data), cols=[i+1]*len(mesh_fig.data))

clean_fig(fig)

# --- Synchronize 2D axis ranges ---
xrange = [-1,1]
yrange = [-1,1]
fig.update_xaxes(range=xrange)
fig.update_yaxes(range=yrange)


# --- Synchronise 3D ranges ---
zrange = [fmin, fmax]
aspectratio = dict(x=1, y=1, z=(zrange[1]-zrange[0])/(xrange[1]-xrange[0]))
scale = 0.8
camera_scaled = camera.copy()
camera_scaled["eye"] = dict(
    x=camera["eye"]["x"]*scale,
    y=camera["eye"]["y"]*scale,
    z=camera["eye"]["z"]*scale,
)
    
for i in range(1, num_frames+1):
    key = f"scene{i}" if i > 1 else "scene"
    fig.update_layout(
        {
            key: dict(
                camera=camera_scaled,
                xaxis=dict(range=xrange),
                yaxis=dict(range=yrange),
                zaxis=dict(range=zrange),
                aspectmode="manual",
                aspectratio=aspectratio,
            )
        }
    )

# --- Layout cleanup ---
fig.update_layout(width=1000, height=400)
fig.update_layout(margin=dict(l=0, r=0, t=0, b=0))

fig.show()
fig.write_image("figs/new_output.pdf", scale=1)


In [ ]:
def labels(row, col):
    if row == 2:
        return r"$t = {:.2f}$".format((col-1)*(t_end/(num_frames-1)))

overpic_labels(fig, labels, 
               stretch_x=1, 
               stretch_y=1,
               offset_y=11,
            #    offset_x=5,
               )

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

num_frames = 5
t_end = 0.4

specs = [
    [{"type": "xy"}]*num_frames,
    [{"type": "scene"}]*num_frames,
]

fig = make_subplots(
    rows=2,
    cols=num_frames,
    specs=specs,
    horizontal_spacing=0.02,
    vertical_spacing=0.02,
)

scatter_size = 3
c_scale = 1.2
fmin = f.to_ambient().min() * c_scale
fmax = f.to_ambient().max() * c_scale
amax=max(abs(fmin), abs(fmax))

# --- Add all traces ---

# Define the space of pairs of functions as a direct sum
function_pairs_space = dg2.function_space + dg2.function_space

# Define the initial value and speed
initial_value = f
initial_speed = dg2.function_space.zeros()
initial_pair = function_pairs_space.pack(initial_value, initial_speed)

# Define the damped wave operator
damping = 0.
operator = block([[zero(dg2.function_space), identity(dg2.function_space)],
                [-dg2.laplacian(0), -damping*identity(dg2.function_space)]])

# Solve the differential operator
t_values = np.linspace(0, t_end, num_frames)
ft = solve_differential_operator(operator, initial_pair, t_values)
values, speeds = ft.unpack()
ft = values.real
for i in range(len(t_values)):
    scatter_fig = plot_scatter_2d(dg2.cache.data_matrix, color = ft[i].to_ambient(), size = scatter_size, amax=amax)
    fig.add_traces(list(scatter_fig.data), rows=[1]*len(scatter_fig.data), cols=[i+1]*len(scatter_fig.data))
    mesh_fig = plot_3d_mesh(data, ft[i], camera=camera, amax=amax)
    fig.add_traces(list(mesh_fig.data), rows=[2]*len(mesh_fig.data), cols=[i+1]*len(mesh_fig.data))
clean_fig(fig)

# --- Synchronize 2D axis ranges ---
xrange = [-1,1]
yrange = [-1,1]
fig.update_xaxes(range=xrange)
fig.update_yaxes(range=yrange)


# --- Synchronise 3D ranges ---
zrange = [fmin, fmax]
aspectratio = dict(x=1, y=1, z=(zrange[1]-zrange[0])/(xrange[1]-xrange[0]))
scale = 0.8
camera_scaled = camera.copy()
camera_scaled["eye"] = dict(
    x=camera["eye"]["x"]*scale,
    y=camera["eye"]["y"]*scale,
    z=camera["eye"]["z"]*scale,
)
    
for i in range(1, num_frames+1):
    key = f"scene{i}" if i > 1 else "scene"
    fig.update_layout(
        {
            key: dict(
                camera=camera_scaled,
                xaxis=dict(range=xrange),
                yaxis=dict(range=yrange),
                zaxis=dict(range=zrange),
                aspectmode="manual",
                aspectratio=aspectratio,
            )
        }
    )

# --- Layout cleanup ---
fig.update_layout(width=1000, height=400)
fig.update_layout(margin=dict(l=0, r=0, t=0, b=0))

fig.show()
fig.write_image("figs/new_output.pdf", scale=1)
